## 0. Setup


## 1. Inspect dataset structure


In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Kaggle-mounted dataset roots
FS_ROOT  = Path('/kaggle/input/competitions/asl-fingerspelling')
SIGN_ROOT = Path('/kaggle/input/competitions/asl-signs')


# Output dirs
OUT = Path('/kaggle/working')
DATA_OUT = OUT / 'data'
LOG_OUT  = OUT / 'logs'
DATA_OUT.mkdir(parents=True, exist_ok=True)
LOG_OUT.mkdir(parents=True, exist_ok=True)

print('Fingerspelling root exists:', FS_ROOT.exists())
print('Isolated signs root exists:', SIGN_ROOT.exists())

In [ ]:
sign_meta = pd.read_csv('/kaggle/input/competitions/asl-signs/train.csv')
all_signs = sign_meta['sign'].value_counts()
print(f'Total signs: {len(all_signs)}, total sequences: {len(sign_meta)}')
print(f'Per-sign mean: {all_signs.mean():.0f}, min: {all_signs.min()}, max: {all_signs.max()}')
print('\nAll 250 signs (sorted alphabetically):')
print(sorted(all_signs.index.tolist()))


In [ ]:
def show_tree(root: Path, depth: int = 1):
    if not root.exists():
        print(f'(missing) {root}')
        return
    for p in sorted(root.iterdir())[:20]:
        print('  ' * 0, p.name, '/' if p.is_dir() else '')
        if p.is_dir() and depth > 0:
            for q in sorted(p.iterdir())[:5]:
                print('  ' * 1, q.name, '/' if q.is_dir() else '')

print('=== asl-fingerspelling ===')
show_tree(FS_ROOT, depth=1)
print('\n=== asl-signs ===')
show_tree(SIGN_ROOT, depth=1)

## 2. Load fingerspelling metadata


In [ ]:
fs_meta = pd.read_csv(FS_ROOT / 'train.csv')
print('Rows:', len(fs_meta))
print('Columns:', list(fs_meta.columns))
fs_meta.head()

In [ ]:
print('Unique participants:', fs_meta['participant_id'].nunique() if 'participant_id' in fs_meta.columns else 'n/a')
print('Unique file_ids:', fs_meta['file_id'].nunique() if 'file_id' in fs_meta.columns else 'n/a')
print('Phrase length stats:')
print(fs_meta['phrase'].str.len().describe())

# First-letter distribution — what we will actually train on for the alphabet classifier
fs_meta['first_letter'] = fs_meta['phrase'].str[0].str.upper()
print('\nFirst-letter distribution (top 10):')
print(fs_meta['first_letter'].value_counts().head(10))

## 3. Peek at one parquet


In [ ]:
fs_parquet_dir = FS_ROOT / 'train_landmarks'
sample_parquet = next(fs_parquet_dir.glob('*.parquet'))
print('Sample file:', sample_parquet.name)

sample_df = pd.read_parquet(sample_parquet)
print('Shape:', sample_df.shape)
print('Total columns:', len(sample_df.columns))

right_hand_cols = [f'{ax}_right_hand_{i}' for i in range(21) for ax in ('x', 'y', 'z')]
left_hand_cols  = [f'{ax}_left_hand_{i}'  for i in range(21) for ax in ('x', 'y', 'z')]

print('Right-hand cols present:', all(c in sample_df.columns for c in right_hand_cols))
print('Left-hand cols present :', all(c in sample_df.columns for c in left_hand_cols))

rh_nan_rate = sample_df[right_hand_cols].isna().all(axis=1).mean()
lh_nan_rate = sample_df[left_hand_cols].isna().all(axis=1).mean()
print(f'Frames with right hand fully missing: {rh_nan_rate:.1%}')
print(f'Frames with left hand fully missing : {lh_nan_rate:.1%}')

sample_df.head(3)

## 4. Helper: per-frame landmark normalization


In [ ]:
def normalize_hand_frame(row_vals: np.ndarray):
    """row_vals: 1-D array of length 63 (x0,y0,z0, x1,y1,z1, ... x20,y20,z20).
    Returns a (63,) float32 vector or None if the frame is unusable."""
    if np.all(np.isnan(row_vals)):
        return None
    coords = np.nan_to_num(row_vals.astype(np.float32), nan=0.0).reshape(21, 3)
    wrist = coords[0].copy()
    coords -= wrist
    max_d = np.linalg.norm(coords, axis=1).max()
    if max_d > 0:
        coords /= max_d
    return coords.flatten()

## 5. Process fingerspelling dataset


In [ ]:
MAX_SEQUENCES   = 5000
FRAMES_PER_SEQ  = 8

rng = np.random.default_rng(42)

fs_sample = fs_meta.sample(min(MAX_SEQUENCES, len(fs_meta)), random_state=42)
fs_sample = fs_sample[fs_sample['phrase'].str.len() > 0].reset_index(drop=True)
print(f'Sampling {len(fs_sample)} sequences from {len(fs_meta)} total.')

wanted_by_file = fs_sample.groupby('file_id')['sequence_id'].apply(set).to_dict()
label_lookup   = dict(zip(fs_sample['sequence_id'], fs_sample['phrase'].str[0].str.upper()))

rows = []
for file_id, wanted_seqs in tqdm(wanted_by_file.items(), desc='parquets'):
    fpath = fs_parquet_dir / f'{file_id}.parquet'
    if not fpath.exists():
        continue
    # sequence_id is the INDEX of the parquet, not a column
    df = pd.read_parquet(fpath, columns=right_hand_cols + left_hand_cols)
    df = df[df.index.isin(wanted_seqs)]
    for seq_id, group in df.groupby(level=0):
        n = len(group)
        if n == 0:
            continue
        idxs = np.linspace(0, n - 1, num=min(FRAMES_PER_SEQ, n)).astype(int)
        sub = group.iloc[idxs]
        for _, row in sub.iterrows():
            rh = row[right_hand_cols].to_numpy(dtype=float)
            lh = row[left_hand_cols].to_numpy(dtype=float)
            feats = normalize_hand_frame(rh)
            if feats is None:
                feats = normalize_hand_frame(lh)
                if feats is not None:
                    feats = feats.copy()
                    feats[0::3] *= -1
            if feats is None:
                continue
            rows.append({
                'sequence_id': seq_id,
                'features': feats.astype(np.float32),
                'label': label_lookup.get(seq_id),
            })

fs_processed = pd.DataFrame(rows)
fs_processed = fs_processed.dropna(subset=['label'])
print('Processed frames:', len(fs_processed))
print('Unique labels   :', fs_processed['label'].nunique())
fs_processed.head()


## 6. Save processed fingerspelling data


In [ ]:
fs_out_path = DATA_OUT / 'fingerspelling_processed.pkl'
fs_processed.to_pickle(fs_out_path)
print('Saved:', fs_out_path, '— size MB:', fs_out_path.stat().st_size / 1e6)

## 7. Process isolated signs dataset


In [ ]:
def normalize_hand_frame(row_vals: np.ndarray):
    if np.all(np.isnan(row_vals)):
        return None
    coords = np.nan_to_num(row_vals.astype(np.float32), nan=0.0).reshape(21, 3)
    wrist = coords[0].copy()
    coords -= wrist
    max_d = np.linalg.norm(coords, axis=1).max()
    if max_d > 0:
        coords /= max_d
    return coords.flatten()

def frames_from_long(df_long, hand_type):
    sub = df_long[df_long['type'] == hand_type]
    if len(sub) == 0:
        return None, 0
    pivot = sub.pivot_table(index='frame', columns='landmark_index', values=['x', 'y', 'z'])
    frames = []
    for f, row in pivot.iterrows():
        coords = np.zeros((21, 3), dtype=float)
        ok = True
        for li in range(21):
            try:
                coords[li] = [row[('x', li)], row[('y', li)], row[('z', li)]]
            except KeyError:
                ok = False
                break
        if not ok:
            continue
        if np.all(np.isnan(coords)):
            continue
        frames.append(coords.flatten())
    if not frames:
        return None, 0
    return np.stack(frames), len(frames)


In [ ]:
sign_meta = pd.read_csv(SIGN_ROOT / 'train.csv')
print('Rows:', len(sign_meta))
print('Columns:', list(sign_meta.columns))
print('Unique signs:', sign_meta['sign'].nunique())
sign_meta.head()

In [ ]:
# Curated list — replace with whatever 30 signs we settle on in Step 2
SELECTED_SIGNS = [
    'hello', 'thankyou', 'please', 'bye',
    'yes', 'no',
    'mom', 'dad', 'brother', 'grandma', 'grandpa',
    'drink', 'sleep', 'go', 'look', 'listen', 'see', 'have', 'finish', 'wait',
    'happy', 'sad', 'hungry', 'sleepy',
    'food', 'water', 'home', 'dog', 'cat',
    'now',
]


# Filter metadata to ONLY these signs — keep ALL their sequences (no cap)
sign_filtered = sign_meta[sign_meta['sign'].isin(SELECTED_SIGNS)].reset_index(drop=True)
print(f'Filtered to {len(SELECTED_SIGNS)} signs, {len(sign_filtered)} sequences total')
print('Per-sign counts:')
print(sign_filtered['sign'].value_counts())

sign_records = []
skipped = 0
for _, row in tqdm(sign_filtered.iterrows(), total=len(sign_filtered), desc='isolated signs'):
    fpath = SIGN_ROOT / row['path']
    if not fpath.exists():
        skipped += 1
        continue
    df_long = pd.read_parquet(fpath)

    rh_arr, rh_n = frames_from_long(df_long, 'right_hand')
    lh_arr, lh_n = frames_from_long(df_long, 'left_hand')

    use_left = lh_n > rh_n
    arr = lh_arr if use_left else rh_arr
    if arr is None:
        skipped += 1
        continue

    frames = []
    for fr in arr:
        v = normalize_hand_frame(fr)
        if v is None:
            continue
        if use_left:
            v = v.copy()
            v[0::3] *= -1
        frames.append(v.astype(np.float32))
    if len(frames) < 5:
        skipped += 1
        continue

    sign_records.append({
        'sequence_id': row['sequence_id'],
        'participant_id': row['participant_id'],
        'features': np.stack(frames),
        'n_frames': len(frames),
        'sign': row['sign'],
    })

sign_processed = pd.DataFrame(sign_records)
print(f'\nKept {len(sign_processed)} sequences, skipped {skipped}')
print(sign_processed['sign'].value_counts())


In [ ]:
sign_out_path = DATA_OUT / 'isolated_signs_processed.pkl'
sign_processed.to_pickle(sign_out_path)
print('Saved:', sign_out_path, '— size MB:', sign_out_path.stat().st_size / 1e6)

## 8. Train / val / test split


In [ ]:
from sklearn.model_selection import train_test_split

fs_seqs = fs_processed['sequence_id'].unique()
fs_train_seq, fs_temp_seq = train_test_split(fs_seqs, test_size=0.2, random_state=42)
fs_val_seq, fs_test_seq   = train_test_split(fs_temp_seq, test_size=0.5, random_state=42)

sign_idx = np.arange(len(sign_processed))
sign_train_idx, sign_temp_idx = train_test_split(sign_idx, test_size=0.2, random_state=42, stratify=sign_processed['sign'])
sign_val_idx, sign_test_idx   = train_test_split(sign_temp_idx, test_size=0.5, random_state=42, stratify=sign_processed['sign'].iloc[sign_temp_idx])

splits = {
    'fingerspelling': {
        'train_sequence_ids': fs_train_seq.tolist(),
        'val_sequence_ids':   fs_val_seq.tolist(),
        'test_sequence_ids':  fs_test_seq.tolist(),
    },
    'isolated_signs': {
        'train_idx': sign_train_idx.tolist(),
        'val_idx':   sign_val_idx.tolist(),
        'test_idx':  sign_test_idx.tolist(),
    },
}
with open(DATA_OUT / 'splits.pkl', 'wb') as f:
    pickle.dump(splits, f)

print(f"FS  splits: train={len(fs_train_seq)}, val={len(fs_val_seq)}, test={len(fs_test_seq)}")
print(f"Sign splits: train={len(sign_train_idx)}, val={len(sign_val_idx)}, test={len(sign_test_idx)}")

# Phase 3.5 — Exploratory Data Analysis


## 9.1 Class distribution


In [ ]:
class_counts = fs_processed['label'].value_counts().sort_index()
print(class_counts)
print(f'\nMax/min ratio: {class_counts.max() / class_counts.min():.1f}x')

fig, ax = plt.subplots(figsize=(14, 5))
class_counts.plot(kind='bar', color='steelblue', edgecolor='black', ax=ax)
ax.set_title('Frames per first-letter label (fingerspelling sample)')
ax.set_ylabel('Count')
ax.set_xlabel('Letter')
plt.tight_layout()
plt.savefig(LOG_OUT / 'class_distribution.png', dpi=150)
plt.show()

## 9.2 Global descriptive statistics


In [ ]:
X = np.stack(fs_processed['features'].values)
feature_names = [f'{ax}_lm{i}' for i in range(21) for ax in ('x', 'y', 'z')]

stats = pd.DataFrame({
    'mean':   X.mean(axis=0),
    'std':    X.std(axis=0),
    'min':    X.min(axis=0),
    'max':    X.max(axis=0),
    'median': np.median(X, axis=0),
    'CV':     X.std(axis=0) / (np.abs(X.mean(axis=0)) + 1e-8),
}, index=feature_names)

print(f'Total samples: {len(X)}, feature dim: {X.shape[1]}')
print('\nSummary of the summary:')
print(stats.describe())
print('\nTop 10 highest-std features:')
print(stats.nlargest(10, 'std')[['mean', 'std', 'CV']])

stats.to_csv(LOG_OUT / 'global_descriptive_stats.csv')

## 9.3 Per-class statistics


In [ ]:
class_means = {}
class_stds  = {}
for label in sorted(fs_processed['label'].unique()):
    mask = fs_processed['label'] == label
    Xc = np.stack(fs_processed.loc[mask, 'features'].values)
    class_means[label] = Xc.mean(axis=0)
    class_stds[label]  = Xc.std(axis=0)

means_df = pd.DataFrame(class_means, index=feature_names).T
stds_df  = pd.DataFrame(class_stds,  index=feature_names).T

means_df.to_csv(LOG_OUT / 'per_class_means.csv')
stds_df.to_csv(LOG_OUT / 'per_class_stds.csv')

print('Per-class mean shape:', means_df.shape, '(classes × features)')
means_df.iloc[:, :6]

## 9.4 Inter-class variance — most discriminative features


In [ ]:
inter_class_var = means_df.var(axis=0).sort_values(ascending=False)
print('Top 20 most discriminative features:')
print(inter_class_var.head(20))

fig, ax = plt.subplots(figsize=(14, 6))
inter_class_var.head(20).plot(kind='bar', color='steelblue', edgecolor='black', ax=ax)
ax.set_title('Top 20 Most Discriminative Landmark Features (inter-class variance)')
ax.set_xlabel('Feature (axis_landmark)')
ax.set_ylabel('Variance across class means')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(LOG_OUT / 'top20_discriminative_features.png', dpi=150)
plt.show()

## 9.5 Correlation heatmap


In [ ]:
corr = np.corrcoef(X.T)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, xticklabels=feature_names, yticklabels=feature_names,
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax,
            cbar_kws={'shrink': 0.7})
ax.set_title('Landmark Feature Correlation Heatmap (63×63)')
plt.xticks(fontsize=6)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.savefig(LOG_OUT / 'correlation_heatmap.png', dpi=150)
plt.show()

# Highly-correlated pair count (informational)
n_high = 0
for i in range(len(feature_names)):
    for j in range(i + 1, len(feature_names)):
        if abs(corr[i][j]) > 0.9:
            n_high += 1
print(f'Pairs with |r| > 0.9: {n_high}')

## 9.6 Box plots of top discriminative features by class


In [ ]:
top5 = inter_class_var.head(5).index.tolist()
top5_idx = [feature_names.index(f) for f in top5]
labels_sorted = sorted(fs_processed['label'].unique())

fig, axes = plt.subplots(1, 5, figsize=(26, 6))
for ax, feat_name, feat_idx in zip(axes, top5, top5_idx):
    data_by_class = []
    for lab in labels_sorted:
        mask = fs_processed['label'] == lab
        vals = np.stack(fs_processed.loc[mask, 'features'].values)[:, feat_idx]
        data_by_class.append(vals)
    ax.boxplot(data_by_class, labels=labels_sorted, showfliers=False)
    ax.set_title(feat_name)
    ax.set_xlabel('Letter')
    ax.tick_params(axis='x', rotation=90, labelsize=7)
fig.suptitle('Distribution of Top 5 Discriminative Features by Letter Class', fontsize=14)
plt.tight_layout()
plt.savefig(LOG_OUT / 'top5_boxplots_by_class.png', dpi=150)
plt.show()

## 10. Deliverables checklist


In [ ]:
for p in sorted((OUT).rglob('*')):
    if p.is_file():
        print(f'{p.relative_to(OUT)}  —  {p.stat().st_size/1e6:.2f} MB')